In [1]:
import glob
import os
import shutil
import json

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm

from tflite_inference import TFLiteInferer

2025-02-25 14:35:43.903071: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1740486943.914514  227880 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1740486943.918002  227880 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-25 14:35:43.929900: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version:  2.18.0


In [2]:
# !pip install tensorflow==2.3

In [3]:
def awb(img):
    # Improve contrast using histogram equalization (adaptive for better results)
    lab = cv2.cvtColor(img.copy(), cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    enhanced_image = cv2.merge((l, a, b))
    enhanced_image = cv2.cvtColor(enhanced_image, cv2.COLOR_LAB2RGB)
    return enhanced_image

In [4]:
BASELINE_CONF_TH = 0.35
EXP_CONF_TH = 0.3
IOU_TH = 0.5

ORIGINAL_H, ORIGIAL_W = 480, 640
W = H = 320
RATIO = W / ORIGIAL_W

inferer_baseline = TFLiteInferer(
    yolo_pth="results/filtered_v3_margin_15/weights/best_saved_model/best_full_integer_quant.tflite",
    head_pth="results/filtered_v3_margin_15/weights/head_saved_model/head_float32.tflite",
    conf_th=BASELINE_CONF_TH,
    iou_th=IOU_TH,
    imgsz=320,
)

inferer_exp = TFLiteInferer(
    yolo_pth="results/filtered_v3_margin_15_reduced_quality/weights/best_saved_model/best_full_integer_quant.tflite",
    head_pth="results/filtered_v3_margin_15_reduced_quality/weights/head_saved_model/head_float32.tflite",
    conf_th=EXP_CONF_TH,
    iou_th=IOU_TH,
    imgsz=320,
)

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [5]:
def draw(img, detections, title=""):
    tmp = img.copy()
    
    cv2.putText(tmp, title, (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
    
    for box in detections:
        cv2.rectangle(tmp, (box[:2] / RATIO).astype(int), ((box[:2].astype(int) + box[2:4]) / RATIO).astype(int) , (255, 0, 0), 1)
        cv2.putText(tmp, f"{box[4]:.2f}", (box[:2] / RATIO).astype(int) + [0, 15], cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

    for p in detections:
        cv2.circle(tmp, (p[-2:] // RATIO).astype(int), 2, (255, 0, 0), 3)
    

    # plt.figure(figsize=(8, 8))
    # plt.imshow(tmp)
    return tmp

In [6]:
vpaths = glob.glob("/home/kyamkovyi/Videos/*.avi")
len(vpaths)

13

In [7]:
for vpath in tqdm(vpaths[:1]):
    frames = []

    cap = cv2.VideoCapture(vpath)
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
        frames[-1] = cv2.cvtColor(frames[-1], cv2.COLOR_BGR2RGB)

    break
    results = []
    for frame in frames:
        dets = inferer.infer(frame)
        res = draw(frame, dets)
        results.append(res)

    cap = cv2.VideoWriter(vpath.replace("/home/kyamkovyi/Videos/", "/home/kyamkovyi/Videos/results/"), cv2.VideoWriter_fourcc(*'DIVX'), 1, (640, 480))
    for res in results:
        cap.write(cv2.cvtColor(res, cv2.COLOR_RGB2BGR))

  0%|          | 0/1 [00:00<?, ?it/s]


In [8]:
res_path = "/home/kyamkovyi/Videos/results_exp_median/"

shutil.rmtree(res_path, ignore_errors=True)
os.mkdir(res_path)

for vpath in tqdm(vpaths):
    frames = []

    cap = cv2.VideoCapture(vpath)
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
        frames[-1] = cv2.cvtColor(frames[-1], cv2.COLOR_BGR2RGB)

    results = []
    for frame in frames:
        full_frame = np.zeros((480, 640 * 2, 3), dtype=np.uint8)
        dets = inferer_baseline.infer(frame)
        res = draw(frame, dets, "gaussian blur model")
        full_frame[:, :640] = res
        
        dets = inferer_exp.infer(frame)
        res = draw(frame, dets, "median blur model")
        full_frame[:, 640:] = res
        
        results.append(full_frame)
    
    # plt.figure(figsize=(18, 8))
    # plt.imshow(results[0])
    # break

    cap = cv2.VideoWriter(vpath.replace("/home/kyamkovyi/Videos/", res_path), cv2.VideoWriter_fourcc(*'DIVX'), 1, (640 * 2, 480))
    for res in results:
        cap.write(cv2.cvtColor(res, cv2.COLOR_RGB2BGR))
    cap.release()

100%|██████████| 13/13 [00:51<00:00,  4.00s/it]


In [35]:
res_path = "/home/kyamkovyi/Videos/results_exp_gray/"

shutil.rmtree(res_path, ignore_errors=True)
os.mkdir(res_path)

for vpath in tqdm(vpaths):
    frames = []

    cap = cv2.VideoCapture(vpath)
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
        frames[-1] = cv2.cvtColor(frames[-1], cv2.COLOR_BGR2RGB)

    results = []
    for frame in frames:
        full_frame = np.zeros((480, 640 * 2, 3), dtype=np.uint8)
        dets = inferer_exp.infer(frame)
        res = draw(frame, dets, "rgb")
        full_frame[:, :640] = res
        
        dets = inferer_exp.infer(cv2.cvtColor(cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY), cv2.COLOR_GRAY2RGB))
        res = draw(frame, dets, "grayscale")
        full_frame[:, 640:] = res
        
        results.append(full_frame)
    
    # plt.figure(figsize=(18, 8))
    # plt.imshow(results[0])
    # break

    cap = cv2.VideoWriter(vpath.replace("/home/kyamkovyi/Videos/", res_path), cv2.VideoWriter_fourcc(*'DIVX'), 1, (640 * 2, 480))
    for res in results:
        cap.write(cv2.cvtColor(res, cv2.COLOR_RGB2BGR))
    cap.release()

100%|██████████| 13/13 [00:51<00:00,  3.99s/it]


In [21]:
frame.shape

(480, 640, 3)